<a href="https://colab.research.google.com/github/MarcoColan01/algo4massivedatasets/blob/main/AMD2425colangelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Finding similar reviews using Jaccard Similarity, Min-Hashing and Locality-Sensitive Hashing (LSH)**

In [1]:
#!pip install kaggle
#!pip install pyspark

# Download of the dataset using Kaggle's API

In [10]:
import pyspark
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import time
from google.colab import files
from collections import Counter

os.environ['KAGGLE_USERNAME'] = "ayeyebrazorf"
os.environ['KAGGLE_KEY'] = "1554a0a7c6d8d35f0029ad0229334515"

!kaggle datasets download -d "mohamedbakhet/amazon-books-reviews" -p .
!unzip -o amazon-books-reviews.zip
!rm -f amazon-books-reviews.zip

start_time = time.time()
spark = pyspark.sql.SparkSession.builder.master("local[*]").appName("similarity").getOrCreate()
sc = spark.sparkContext

books_rating_df = spark.read.csv(
    "Books_rating.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape="\""
)
'''
books_data_df = spark.read.csv(
    "books_data.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape="\""
)
'''

Dataset URL: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews
License(s): CC0-1.0
100% 1.06G/1.06G [00:21<00:00, 42.7MB/s]
100% 1.06G/1.06G [00:21<00:00, 52.5MB/s]
Archive:  amazon-books-reviews.zip
  inflating: Books_rating.csv        
  inflating: books_data.csv          


'\nbooks_data_df = spark.read.csv(\n    "books_data.csv",\n    header=True,\n    inferSchema=True,\n    multiLine=True,\n    escape="""\n)\n'

# First step: sample generation

The Amazon Review Book dataset consists of two tables:
- books_rating: which contains information for approximately 3 million reviews of books
- books_data: which contains information for approximately 212000 books

**The task is to find similar reviews using Jaccard Similarity and a combination of Min-Hashing and LSH to reduce the computational complexity.**

For the work performed, only the *books_rating* dataset was considered because it contains information on reviews.
After downloading the dataset using Kaggle, books_rating.csv is available, which consists of 10 columns. At this point, the first steps are to generate a reasonably sized sample (since we don't have a real execution cluster) and retain only the columns necessary to perform the work. The sample is generated randomly by taking 30,000 reviews, and the following columns are extracted from this sample:
- ID: the book ID
- Title: the book title
- review/text: the entire text of the review

A filter was then applied to discard any reviews with null values ​​or empty ones.

In [13]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import re, math, time

SAMPLE_SIZE = 30000

base_df = (
    books_rating_df
    .select(*["Id", "User_id", "Title", "review/text"])
    .filter((F.col("review/text").isNotNull()) & (F.length(F.col("review/text")) > 0))
)

t0 = time.time()
total = base_df.count()
frac = min(1.0, (SAMPLE_SIZE / max(total, 1)) * 1.20)

subsample_df = (
    base_df
    .sample(False, frac)
    .limit(SAMPLE_SIZE)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

subsample_df.show(10, truncate=False)

+----------+--------------+-------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Second step: normalization and shingling of the text

After the generation of the text, the next step is to apply a normalization of the text (the review/text column) and create the shingles for each review.

A shingle is a k-gram (or k-set) of atomic information, such that the individual characters. For the work done, the choice of the parameter k has been 5 since there are both rather short and rather long reviews, and choosing an ideal k parameter for very long texts (for example k=9) would have instead penalized the shorter texts, so k=5 is a good compromise.

Speaking instead of the text normalization phase, this one has involved two different steps:
- Converting the entire text to lowercase
- Rmoving punctuation marks and non-alphanumeric characters, including spaces.

In [14]:
_ws_re = re.compile(r"\s+")
_punct_re = re.compile(r"[^a-z0-9]+")

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    t = text.lower()
    t = _ws_re.sub(" ", t)
    t = _punct_re.sub("", t)
    return t

#Normalization of the review/text colunn content
t0 = time.time()
normalized_rdd = (
    subsample_df
    .select(F.col("Id").alias("book_id"), F.col("User_ID").alias("user_id"), F.col("review/text"))
    .rdd
    .mapPartitions(lambda it: (
        ((row["book_id"], row["user_id"]), normalize_text(row["review/text"]))
        for row in it
    ))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

In addition to generating the shingles, in this step the generated shingles are also encoded using 64-bit encoding.

In [15]:
K_SHINGLE = 5
FNV64_OFFSET = 1469598103934665603
FNV64_PRIME = 1099511628211
MASK64 = (1 << 64) - 1

def fnv1a_64(s: str) -> int:
    h = FNV64_OFFSET
    for ch in s:
        h ^= ord(ch)
        h = (h * FNV64_PRIME) & MASK64
    return h

def generate_encoded_shingles(text_norm: str, k: int):
    L = len(text_norm)
    if L < k:
        return
    for i in range(L - k + 1):
        yield fnv1a_64(text_norm[i:i+k])

reviews_rdd = (
    normalized_rdd
    .mapPartitions(lambda it: (
        (K, set(generate_encoded_shingles(tnorm, K_SHINGLE)))
        for K, tnorm in it
    ))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Third step: Min-Hashing
Min-hashing is a data compression method that allows us to efficiently estimate Jaccard similarity between sets without having to work directly with the full sets or characteristic matrix.


In [16]:
H = 128
SEED_HASH = 1337
MERSENNE_P = (1 << 61) - 1

nonempty_rdd = reviews_rdd.filter(lambda x: len(x[1]) > 0).persist(StorageLevel.MEMORY_AND_DISK)

rng = np.random.RandomState(SEED_HASH)
a_params = rng.randint(1, MERSENNE_P, size=H, dtype=np.uint64)
b_params = rng.randint(0, MERSENNE_P, size=H, dtype=np.uint64)

broadcast_params = sc.broadcast((a_params, b_params, np.uint64(MERSENNE_P), H))

def minhash(iter_rows):
    a, b, p, H = broadcast_params.value
    for K, shingle_set in iter_rows:
        sig = np.full(H, p - 1, dtype=np.uint64)
        for x in shingle_set:
            hx = (a * np.uint64(x) + b) % p
            sig = np.minimum(sig, hx)
        yield (K, sig)

signatures_rdd = nonempty_rdd.mapPartitions(minhash).persist(StorageLevel.MEMORY_AND_DISK)
sig_count = signatures_rdd.count()
print(f"[MINHASH] Firme calcolate: {sig_count} doc | time={time.time()-t0:.2f}s")

[MINHASH] Firme calcolate: 30000 doc | time=243.72s


# Fourth step: Locality-Sensitive Hashing

In [17]:
B = 32
R = 4

MAX_BUCKET_SIZE = 2000
NUM_PARTITIONS = min(signatures_rdd.getNumPartitions() * 2, 1024)

def digest_subsignature(subsig):
    acc = FNV64_OFFSET
    for v in subsig:
        acc ^= (int(v) & MASK64)
        acc = (acc * FNV64_PRIME) & MASK64
    return acc

def emit_band_keys_partition(iter_rows):
    for K, sig in iter_rows:
        for b_idx in range(B):
            start = b_idx * R
            subsig = sig[start:start+R]
            key = (b_idx, digest_subsignature(subsig))
            yield (key, K)

pairs_by_band = (
    signatures_rdd
    .mapPartitions(emit_band_keys_partition)
    .partitionBy(NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

bucket_docs_rdd = (
    pairs_by_band
    .combineByKey(
        lambda v: [v],
        lambda acc, v: (acc.append(v) or acc),
        lambda a, b: (a.extend(b) or a),
        numPartitions=NUM_PARTITIONS
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

def pairs_from_ids(ids_list):
    uniq_sorted = sorted(set(ids_list))
    m = len(uniq_sorted)
    if m < 2:
        return
    m_cap = m if m <= MAX_BUCKET_SIZE else MAX_BUCKET_SIZE
    for i in range(m_cap):
        a = uniq_sorted[i]
        for j in range(i+1, m_cap):
            b = uniq_sorted[j]
            yield (a, b) if a < b else (b, a)

candidate_pairs_rdd = (
    bucket_docs_rdd
    .flatMap(lambda kv: pairs_from_ids(kv[1]))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

unique_candidate_pairs_rdd = (
    candidate_pairs_rdd
    .distinct()
    .persist(StorageLevel.MEMORY_AND_DISK)
)

pairs_by_band.unpersist()
candidate_pairs_rdd.unpersist()


PythonRDD[159] at RDD at PythonRDD.scala:53

# Fifth step: Jaccard similarity

In [ ]:
OUTPUT_FILE = "similar_reviews.txt"
EPS = 1e-12
MINIMUM_SIMILARITY_THRESHOLD = 0.40
t_all = time.time()

pairs = (
    unique_candidate_pairs_rdd
    .map(lambda ab: (ab[0], ab[1]) if ab[0] < ab[1] else (ab[1], ab[0]))
    .distinct()
)

ids_needed = pairs.flatMap(lambda ab: [ab[0], ab[1]]).distinct()
ids_needed_kv = ids_needed.map(lambda i: (i, None))

needed_sets_rdd = (
    reviews_rdd
    .join(ids_needed_kv)
    .mapValues(lambda v: v[0])
)

pairs_by_A = pairs.groupByKey()
sideA = (
    pairs_by_A
    .join(needed_sets_rdd)
    .flatMap(lambda kv: [((kv[0], b), ('A', kv[1][1])) for b in kv[1][0]])
)

pairs_by_B = pairs.map(lambda ab: (ab[1], ab[0])).groupByKey()
sideB = (
    pairs_by_B
    .join(needed_sets_rdd)
    .flatMap(lambda kv: [((a, kv[0]), ('B', kv[1][1])) for a in kv[1][0]])
)

def compute_jaccard(kv):
    (a, b), parts = kv
    setA = None
    setB = None
    for tag, s in parts:
        if tag == 'A':
            setA = s
        elif tag == 'B':
            setB = s
    if setA is None or setB is None:
        return None
    inter = len(setA & setB)
    if inter == 0:
        return None
    uni = len(setA | setB)
    j = inter / uni if uni > 0 else 1.0
    if (j + EPS) >= MINIMUM_SIMILARITY_THRESHOLD and (1.0 - j) > EPS:
        return (a, b, j)
    return None

joined_parts = sideA.union(sideB).groupByKey()
jaccard_hits_rdd = joined_parts.map(compute_jaccard).filter(lambda x: x is not None)

final_pairs = jaccard_hits_rdd.collect()

if final_pairs:
    final_ids = list({i for a, b, _ in final_pairs for i in (a, b)})
    texts_map = dict(
        subsample_df
        .filter(F.col("Id").isin(final_ids))
        .select(F.col("Id").cast("string").alias("Id"), F.col("review/text").cast("string").alias("text"))
        .rdd
        .map(lambda r: (r["Id"], r["text"]))
        .collect()
    )
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for a, b, jac in final_pairs:
            textA = texts_map.get(a, "")
            textB = texts_map.get(b, "")
            f.write(f"Review 1: {textA}\n")
            f.write(f"Review 2: {textB}\n")
            f.write(f"Jaccard similarity: {jac:.4f}\n")
            f.write("\n" + "="*80 + "\n\n")
else:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(f"No pair with at least {MINIMUM_SIMILARITY_THRESHOLD} ≤ Jaccard < 1\n")